In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Ensure numeric where appropriate
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Aggregate metric.csv by (cmdb_id, kpi_name) using the full series (no time filtering)
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        min='min',
        max='max'
    )
    .reset_index()
)

metric_summary = metric_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'kpi_name', 'count', 'p50', 'p95', 'min', 'max']
]

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        min='min',
        max='max'
    )
    .reset_index()
)

trace_summary = trace_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'trace_name', 'count', 'p50', 'p95', 'min', 'max']
]

# Aggregate log.csv by (cmdb_id, log_name) computing count and p95
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

log_summary = log_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'log_name', 'count', 'p95']
]

# Prepare error_logs sample (up to 20 rows) with timestamp converted to UTC+8
if 'timestamp' in df_error.columns:
    # convert seconds -> datetime UTC then to Asia/Shanghai
    df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_sample = df_error[['timestamp', 'cmdb_id', 'message']].head(20)

# Display compact summaries (variables as outputs)
metric_summary, trace_summary, log_summary, error_sample

```
Out[1]:
```
```python
# Summarize the previous aggregation results in plain English as a single compact string.
# Reuse existing variables: metric_agg, trace_agg, log_agg, metric_summary, trace_summary, log_summary, error_sample

summary_lines = []

# Metrics summary
metric_groups = metric_agg.shape[0] if 'metric_agg' in globals() else metric_summary.shape[0]
summary_lines.append(f"Metrics: {metric_groups} (cmdb_id, kpi_name) groups aggregated. Top 5 by sample count:")
for r in metric_summary.head(5).itertuples(index=False):
    # metric_summary columns: cmdb_id, kpi_name, count, p50, p95, min, max
    cmdb, kpi, cnt, p50, p95, vmin, vmax = r
    summary_lines.append(f" - {cmdb} | {kpi} | count={int(cnt)} | p95={p95:.4g}")

# Traces summary
trace_groups = trace_agg.shape[0] if 'trace_agg' in globals() else trace_summary.shape[0]
summary_lines.append(f"Traces: {trace_groups} (cmdb_id, trace_name) groups aggregated. Top 5 by sample count:")
for r in trace_summary.head(5).itertuples(index=False):
    # trace_summary columns: cmdb_id, trace_name, count, p50, p95, min, max
    cmdb, tname, cnt, p50, p95, vmin, vmax = r
    summary_lines.append(f" - {cmdb} | {tname} | count={int(cnt)} | p95={p95:.4g}")

# Logs summary
log_groups = log_agg.shape[0] if 'log_agg' in globals() else log_summary.shape[0]
summary_lines.append(f"Logs: {log_groups} (cmdb_id, log_name) groups aggregated. Top 5 by sample count:")
for r in log_summary.head(5).itertuples(index=False):
    # log_summary columns: cmdb_id, log_name, count, p95
    cmdb, lname, cnt, p95 = r
    summary_lines.append(f" - {cmdb} | {lname} | count={int(cnt)} | p95={p95:.4g}")

# Error logs sample
if 'error_sample' in globals() and not error_sample.empty:
    total_errors = len(error_sample)
    # show earliest timestamp in sample and number of rows returned earlier (up to 20)
    earliest = error_sample['timestamp'].min()
    latest = error_sample['timestamp'].max()
    summary_lines.append(f"Error logs: sample returned {total_errors} rows (timestamps shown are in UTC+8). Range in sample: {earliest} to {latest}.")
else:
    summary_lines.append("Error logs: no rows returned in sample (error_logs.csv appears empty).")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           p50           p95           min           max
1587  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
15        IG01                             OSLinux-CPU_CPU_CPUWio     31  2.080000e-02  4.864500e-01  8.300000e-03  7.856990e-01
1674  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.035035e+06  6.035846e+06  6.034130e+06  6.035936e+06
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
14        IG01                        OSLinux-CPU_CPU_CPUUserTime     31  2.520160e+01  2.525359e+01  2.516210e+01  2.527850e+01
...        ...                                                ...    ...           ...           ...           ...           ...
1402  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+00  3.500000e+00  0.000000e+00  6.000000e+00
1403  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
1392  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
103       IG02                         OSLinux-CPU_CPU_CPUSysTime     31  3.658000e-01  4.530200e-01  2.951000e-01  5.528310e-01
1393  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 7 columns],       cmdb_id                     trace_name  count          p50          p95         min          max
257  dockerB2           trace.self.row_count     30  1235.000000  1828.200000  732.000000  1938.000000
219  dockerA2       trace.self.duration_mean     30     0.015160     0.099113    0.005509     0.205081
220  dockerA2        trace.self.duration_p95     30     0.008950     0.391495    0.001000     1.019300
221  dockerA2           trace.self.row_count     30  1005.500000  1424.300000  636.000000  1439.000000
201  dockerA1       trace.self.duration_mean     30     0.028082     0.106323    0.015566     0.160735
..        ...                            ...    ...          ...          ...         ...          ...
165  Tomcat03       trace.self.duration_mean     30     0.040130     0.238777    0.016729     0.259760
166  Tomcat03        trace.self.duration_p95     30     0.045875     1.802425    0.024000     2.026200
167  Tomcat03           trace.self.row_count     30  1168.500000  1482.100000  937.000000  1599.000000
168  Tomcat03    trace.to_MG01.duration_mean     30     0.250712     1.877436    0.098433     1.950949
89       MG02  trace.from_Tomcat04.row_count     30    92.000000   111.650000   69.000000   125.000000

[50 rows x 7 columns],      cmdb_id          log_name  count       p95
0   Tomcat01   log.error_count     31    0.0000
2   Tomcat01     log.row_count     31  223.0000
3   Tomcat02   log.error_count     31    0.0000
5   Tomcat02     log.row_count     31  215.5000
6   Tomcat03   log.error_count     31    0.0000
11  Tomcat04     log.row_count     31  212.5000
9   Tomcat04   log.error_count     31    0.0000
8   Tomcat03     log.row_count     31  218.5000
14  apache02   log.error_count     31    0.0000
13  apache01     log.row_count     31  421.0000
15  apache02     log.row_count     31  417.5000
12  apache01   log.error_count     31    0.0000
10  Tomcat04  log.gc_pause_max     11    0.0950
7   Tomcat03  log.gc_pause_max     10    0.0955
1   Tomcat01  log.gc_pause_max     10    0.0900
4   Tomcat02  log.gc_pause_max     10    0.0800, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse existing variables: df_metric, metric_agg, tz if available
# Ensure tz variable for UTC+8
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Allowed root cause components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric rows to allowed components
df_metric_allowed = df_metric[df_metric['cmdb_id'].isin(allowed)].copy()

# convert timestamp to timezone-aware datetime in UTC+8
df_metric_allowed['dt'] = pd.to_datetime(df_metric_allowed['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window in UTC+8 (inclusive)
start = tz.localize(datetime(2021, 3, 4, 13, 30))
end = tz.localize(datetime(2021, 3, 4, 14, 0))

# Filter to time window
df_window = df_metric_allowed[(df_metric_allowed['dt'] >= start) & (df_metric_allowed['dt'] <= end)].copy()

# Attach global p95 threshold from metric_agg (precomputed on full series)
# Ensure metric_agg exists and has p95 column
if 'metric_agg' not in globals():
    raise RuntimeError("metric_agg not found in environment (required global thresholds).")

# Keep only needed columns from metric_agg
thresholds = metric_agg[['cmdb_id', 'kpi_name', 'p95']].copy()
thresholds = thresholds.rename(columns={'p95': 'p95_threshold'})

# Merge thresholds onto windowed data
df_merged = pd.merge(df_window, thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Exclude rows without a global p95 threshold (can't evaluate anomaly)
df_merged_valid = df_merged[~df_merged['p95_threshold'].isna()].copy()

# Mark anomaly if value >= p95_threshold (per instruction)
df_merged_valid['is_anomaly'] = df_merged_valid['value'] >= df_merged_valid['p95_threshold']

# Extract anomaly rows
anomaly_rows = df_merged_valid[df_merged_valid['is_anomaly']].copy()

# Aggregate anomalies by cmdb_id and kpi_name
if not anomaly_rows.empty:
    anomalies_agg = (
        anomaly_rows
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'size'),
            earliest_anomaly_timestamp=('dt', 'min'),
            max_value_within_window=('value', 'max'),
            p95_threshold=('p95_threshold', 'first')
        )
        .sort_values('anomaly_count', ascending=False)
    )
else:
    anomalies_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count',
                                          'earliest_anomaly_timestamp','max_value_within_window','p95_threshold'])

# Prepare top 20 compact output
anomalies_top20 = anomalies_agg.head(20).copy()

# Summary: total anomalous groups and earliest anomaly across all groups
total_anomalous_groups = int(anomalies_agg.shape[0])
earliest_overall = anomalies_agg['earliest_anomaly_timestamp'].min() if total_anomalous_groups > 0 else None

# Compact overall summary as a small DataFrame
overall_summary = pd.DataFrame([{
    'total_anomalous_groups': total_anomalous_groups,
    'earliest_anomaly_timestamp_UTC+8': earliest_overall
}])

# Display compact results (variables)
anomalies_top20, overall_summary

```
Out[2]:
```
```python
# Summarize the anomaly aggregation results in plain English as a single compact string.
# Reuse existing variables from prior steps: anomalies_top20, overall_summary

# Prepare top 5 summary lines
top5 = anomalies_top20.head(5)
lines = []
total = int(overall_summary.at[0, 'total_anomalous_groups']) if 'overall_summary' in globals() else 0
earliest = overall_summary.at[0, 'earliest_anomaly_timestamp_UTC+8'] if 'overall_summary' in globals() else None
earliest_str = str(earliest) if pd.notna(earliest) else "None"

lines.append(f"Summary: Found {total} anomalous (cmdb_id, kpi_name) groups during 2021-03-04 13:30–14:00 (UTC+8).")
lines.append(f"Earliest anomaly timestamp (UTC+8): {earliest_str}.")
lines.append("Top 5 groups by anomaly_count (cmdb_id | kpi_name | anomaly_count | max_value | p95_threshold):")

for r in top5.itertuples(index=False):
    cmdb = getattr(r, 'cmdb_id')
    kpi = getattr(r, 'kpi_name')
    cnt = int(getattr(r, 'anomaly_count'))
    vmax = float(getattr(r, 'max_value_within_window'))
    p95 = float(getattr(r, 'p95_threshold'))
    # Keep KPI names compact if very long
    kpi_short = (kpi[:80] + '...') if len(kpi) > 80 else kpi
    lines.append(f" - {cmdb} | {kpi_short} | count={cnt} | max={vmax:g} | p95={p95:g}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_within_window  p95_threshold
1623  apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-04 13:30:00+08:00             7.000000e+00   7.000000e+00
1354  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1355  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1356  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1357  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-04 13:30:00+08:00             1.000000e+02   1.000000e+02
1014   Redis02                  redis-Redis_6379_Redis  (loading)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1000   Redis02              redis-Redis_6379_Redis  (aof_enabled)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1001   Redis02  redis-Redis_6379_Redis  (aof_rewrite_in_progress)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1002   Redis02    redis-Redis_6379_Redis  (aof_rewrite_scheduled)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1003   Redis02          redis-Redis_6379_Redis  (blocked_clients)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1004   Redis02  redis-Redis_6379_Redis  (client_biggest_input_...             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1008   Redis02             redis-Redis_6379_Redis  (evicted_keys)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1009   Redis02             redis-Redis_6379_Redis  (expired_keys)             31  2021-03-04 13:30:00+08:00             1.000000e+00   1.000000e+00
1006   Redis02        redis-Redis_6379_Redis  (connected_clients)             31  2021-03-04 13:30:00+08:00             4.020000e+02   4.020000e+02
1007   Redis02         redis-Redis_6379_Redis  (connected_slaves)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1013   Redis02         redis-Redis_6379_Redis  (latest_fork_usec)             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
1137  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
986    Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
987    Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-04 13:30:00+08:00             0.000000e+00   0.000000e+00
90        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-04 13:30:00+08:00             2.040136e+09   2.040136e+09,    total_anomalous_groups earliest_anomaly_timestamp_UTC+8
0                    1631        2021-03-04 13:30:00+08:00)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse anomalies_agg produced previously
if 'anomalies_agg' not in globals():
    raise RuntimeError("anomalies_agg not found in environment. Run previous anomaly detection steps first.")

# Work on a copy
anoms = anomalies_agg.copy()

# Ensure numeric columns
anoms['max_value_within_window'] = pd.to_numeric(anoms['max_value_within_window'], errors='coerce').fillna(0)
anoms['p95_threshold'] = pd.to_numeric(anoms['p95_threshold'], errors='coerce').fillna(0)

# Compute deviation ratio with special handling:
# if p95 == 0 and max_value > 0 -> inf
# if p95 == 0 and max_value == 0 -> 1
def compute_ratio(row):
    p95 = row['p95_threshold']
    vmax = row['max_value_within_window']
    if p95 == 0:
        return np.inf if vmax > 0 else 1.0
    else:
        return vmax / p95

anoms['deviation_ratio'] = anoms.apply(compute_ratio, axis=1)

# For each cmdb_id compute required aggregations
# 1) anomalous_kpi_groups = count distinct kpi_name
# 2) total_anomaly_points = sum anomaly_count
# 3) earliest_anomaly_timestamp_UTC+8 = min earliest_anomaly_timestamp
# 4) max_deviation_ratio = max deviation_ratio
# 5) top_kpis = up to 3 kpi_name values with highest individual deviation_ratio (semicolon-separated)
def top_kpis_func(df, topn=3):
    df_sorted = df.sort_values('deviation_ratio', ascending=False)
    topk = df_sorted['kpi_name'].head(topn).astype(str).tolist()
    return ";".join(topk)

agg = (
    anoms
    .groupby('cmdb_id', as_index=False)
    .agg(
        anomalous_kpi_groups=('kpi_name', 'nunique'),
        total_anomaly_points=('anomaly_count', 'sum'),
        earliest_anomaly_timestamp_UTC_plus8=('earliest_anomaly_timestamp', 'min'),
        max_deviation_ratio=('deviation_ratio', 'max')
    )
)

# Compute top_kpis separately using groupby.apply
top_kpis_series = anoms.groupby('cmdb_id').apply(top_kpis_func).rename('top_kpis').reset_index()

# Merge
cmdb_agg = pd.merge(agg, top_kpis_series, on='cmdb_id', how='left')

# Sort by max_deviation_ratio desc and take top 20
cmdb_agg_top20 = cmdb_agg.sort_values('max_deviation_ratio', ascending=False).head(20)

# Keep output compact: select and order columns as requested
cmdb_agg_top20 = cmdb_agg_top20[
    ['cmdb_id', 'anomalous_kpi_groups', 'total_anomaly_points', 'max_deviation_ratio',
     'earliest_anomaly_timestamp_UTC_plus8', 'top_kpis']
].rename(columns={'earliest_anomaly_timestamp_UTC_plus8': 'earliest_anomaly_timestamp_UTC+8'})

# Display the compact result (variable output)
cmdb_agg_top20

```
Out[3]:
```
```python
# Summarize the per-component aggregation results (reuse cmdb_agg_top20)
lines = []
n_components = cmdb_agg_top20.shape[0]
overall_earliest = cmdb_agg_top20['earliest_anomaly_timestamp_UTC+8'].min() if n_components>0 else None
lines.append(f"Found {n_components} components with detected anomalies during the window. Earliest anomaly: {overall_earliest} (UTC+8).")
lines.append("Top components by max_deviation_ratio (showing up to 6):")

for r in cmdb_agg_top20.head(6).itertuples(index=False):
    cmdb = r.cmdb_id
    groups = int(r.anomalous_kpi_groups)
    points = int(r.total_anomaly_points)
    ratio = float(r.max_deviation_ratio)
    earliest = r._4  # earliest_anomaly_timestamp_UTC+8 column position
    topk = r.top_kpis
    # keep top_kpis compact
    topk_short = (topk[:160] + '...') if isinstance(topk, str) and len(topk) > 160 else topk
    lines.append(f" - {cmdb}: anomalous_kpi_groups={groups}, total_anomaly_points={points}, max_deviation_ratio={ratio:g}, earliest={earliest}, top_kpis={topk_short}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  anomalous_kpi_groups  total_anomaly_points  max_deviation_ratio earliest_anomaly_timestamp_UTC+8                                           top_kpis
4    Mysql01                   224                  2223            20.000000        2021-03-04 13:30:00+08:00  Mysql-MySQL_3306_MaxTrxRowsModified;Mysql-MySQ...
5    Mysql02                   224                  1938            20.000000        2021-03-04 13:30:00+08:00  Mysql-MySQL_3306_Innodb pages read;Mysql-MySQL...
7    Redis02                   109                  1376             4.000000        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...
2       MG01                    92                   630             3.626093        2021-03-04 13:30:00+08:00  OSLinux-CPU_CPU_CPUWio;OSLinux-OSLinux_LOCALDI...
11  Tomcat04                   111                  1004             3.597635        2021-03-04 13:30:00+08:00  OSLinux-CPU_CPU_CPUWio;OSLinux-OSLinux_LOCALDI...
12  apache01                    80                   581             3.333333        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
13  apache02                    80                   616             3.333333        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...
6    Redis01                   114                  1384             2.914282        2021-03-04 13:30:00+08:00  OSLinux-CPU_CPU-2_SingleCpuidle;OSLinux-CPU_CP...
1       IG02                    89                   650             2.857143        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps...
0       IG01                    90                   669             2.000000        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps...
10  Tomcat03                   109                  1028             2.000000        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps...
9   Tomcat02                   109                  1045             2.000000        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps...
3       MG02                    92                   580             1.818182        2021-03-04 13:30:00+08:00  OSLinux-CPU_CPU-2_SingleCpuidle;OSLinux-OSLinu...
8   Tomcat01                   108                   930             1.818182        2021-03-04 13:30:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps...```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# Reuse anomalies_agg if available
if 'anoms' in globals():
    df = anoms.copy()
elif 'anomalies_agg' in globals():
    df = anomalies_agg.copy()
else:
    raise RuntimeError("No anomalies table found in workspace (anoms or anomalies_agg required).")

# Ensure timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Filter to Mysql01 and Mysql02
mysql_df = df[df['cmdb_id'].isin(['Mysql01', 'Mysql02'])].copy()

# Ensure numeric columns
mysql_df['max_value_within_window'] = pd.to_numeric(mysql_df['max_value_within_window'], errors='coerce').fillna(0)
mysql_df['p95_threshold'] = pd.to_numeric(mysql_df['p95_threshold'], errors='coerce').fillna(0)

# Ensure earliest_anomaly_timestamp is timezone-aware UTC+8
if 'earliest_anomaly_timestamp' in mysql_df.columns:
    mysql_df['earliest_anomaly_timestamp'] = pd.to_datetime(mysql_df['earliest_anomaly_timestamp']).dt.tz_convert(tz)

# Compute deviation_ratio with special rules
p95 = mysql_df['p95_threshold'].values
vmax = mysql_df['max_value_within_window'].values
ratio = np.where(p95 == 0, np.where(vmax > 0, np.inf, 1.0), vmax / p95)
mysql_df['deviation_ratio'] = ratio

# Select and sort top 20 by deviation_ratio desc
cols = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp',
        'max_value_within_window', 'p95_threshold', 'deviation_ratio']
mysql_anoms_top20 = mysql_df[cols].sort_values('deviation_ratio', ascending=False).head(20)

# Display compact result
mysql_anoms_top20

```
Out[4]:
```
```python
# Summarize Mysql01/Mysql02 anomaly details in plain English (compact)
try:
    df = mysql_anoms_top20.copy()
except NameError:
    # Fallback if variable not present
    df = None

if df is None or df.empty:
    summary = "No anomaly rows for Mysql01 or Mysql02 were found in the previously detected anomalies table."
else:
    n_rows = int(df.shape[0])
    earliest = df['earliest_anomaly_timestamp'].min()
    # Prepare top 5 highest deviation KPIs
    top5 = df.sort_values('deviation_ratio', ascending=False).head(5)
    lines = [
        f"Summary for Mysql01 and Mysql02 (window 2021-03-04 13:30–14:00 UTC+8):",
        f"- Returned rows: {n_rows}. Earliest anomaly timestamp: {earliest}.",
        "- Key findings:"
    ]
    for r in top5.itertuples(index=False):
        cmdb = r.cmdb_id
        kpi = r.kpi_name
        cnt = int(r.anomaly_count)
        vmax = float(r.max_value_within_window)
        p95 = float(r.p95_threshold)
        ratio = r.deviation_ratio
        lines.append(f"  • {cmdb} | {kpi} | anomalies={cnt} | max={vmax:g} | p95={p95:g} | ratio={ratio:g}")
    lines.append("- Observations: Several MySQL KPIs (InnoDB reads, Max trx/lock metrics and related counters) show very high deviation ratios (many at 20x). Most top anomalies are single points (anomaly_count=1) occurring between ~13:31 and 13:50 UTC+8. Other notable elevated metrics include Handler Read Next (ratio ~12.3) and multiple disk-related KPIs (ratios ~3–4).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_within_window  p95_threshold  deviation_ratio
658  Mysql02                 Mysql-MySQL_3306_Innodb pages read              1  2021-03-04 13:50:00+08:00                 0.016700       0.000835        20.000000
445  Mysql01               Mysql-MySQL_3306_Max trx rows locked              1  2021-03-04 13:32:00+08:00                 1.000000       0.050000        20.000000
486  Mysql01         Mysql-MySQL_3306_max trx lock memory bytes              1  2021-03-04 13:32:00+08:00              1136.000000      56.800000        20.000000
669  Mysql02               Mysql-MySQL_3306_Max trx rows locked              1  2021-03-04 13:35:00+08:00                 1.000000       0.050000        20.000000
643  Mysql02                  Mysql-MySQL_3306_Innodb data read              1  2021-03-04 13:50:00+08:00               273.066700      13.653335        20.000000
644  Mysql02                 Mysql-MySQL_3306_Innodb data reads              1  2021-03-04 13:50:00+08:00                 0.016700       0.000835        20.000000
636  Mysql02          Mysql-MySQL_3306_Innodb buffer pool reads              1  2021-03-04 13:50:00+08:00                 0.016700       0.000835        20.000000
696  Mysql02                        Mysql-MySQL_3306_Sort Range              1  2021-03-04 13:44:00+08:00                 0.266700       0.013335        20.000000
433  Mysql01              Mysql-MySQL_3306_Innodb pages created              1  2021-03-04 13:49:00+08:00                 0.016700       0.000835        20.000000
447  Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified              1  2021-03-04 13:32:00+08:00                 1.000000       0.050000        20.000000
616  Mysql02                 Mysql-MySQL_3306_Handler Read Next              1  2021-03-04 13:39:00+08:00              8006.683300     649.028300        12.336416
773  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1  2021-03-04 13:48:00+08:00                 5.000000       1.250000         4.000000
541  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-04 13:38:00+08:00                 0.400000       0.100000         4.000000
625  Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time              1  2021-03-04 13:32:00+08:00                24.827600       7.416380         3.347671
549  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1  2021-03-04 13:37:00+08:00                 5.000000       1.500000         3.333333
550  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              1  2021-03-04 13:31:00+08:00                 2.400000       0.720000         3.333333
543  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead              1  2021-03-04 13:58:00+08:00                27.200000       8.346667         3.258786
542  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps              1  2021-03-04 13:58:00+08:00                 1.883300       0.588323         3.201132
765  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-04 13:31:00+08:00                 7.200000       2.520000         2.857143
784  Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps              1  2021-03-04 14:00:00+08:00                 0.966667       0.349188         2.768325```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing variables if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    trace_agg
except NameError:
    raise RuntimeError("trace_agg (precomputed global thresholds) not found in environment.")

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# service components list
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02']

# Filter trace rows by component OR trace_name contains 'Mysql' (case-insensitive)
mask_comp = df_trace['cmdb_id'].isin(components)
mask_mysql = df_trace['trace_name'].str.contains('mysql', case=False, na=False)
df_filtered = df_trace[mask_comp | mask_mysql].copy()

# convert timestamp to timezone-aware datetime in UTC+8
df_filtered['dt'] = pd.to_datetime(df_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# incident window (inclusive) in UTC+8
start = tz.localize(datetime(2021, 3, 4, 13, 30))
end = tz.localize(datetime(2021, 3, 4, 14, 0))
df_window = df_filtered[(df_filtered['dt'] >= start) & (df_filtered['dt'] <= end)].copy()

# attach global p95 threshold from trace_agg (which was computed on full series)
# ensure trace_agg has 'p95' column
if 'p95' not in trace_agg.columns:
    raise RuntimeError("trace_agg does not contain 'p95' column (required global thresholds).")

thresholds = trace_agg[['cmdb_id', 'trace_name', 'p95']].rename(columns={'p95': 'p95_threshold'})
df_merged = pd.merge(df_window, thresholds, on=['cmdb_id', 'trace_name'], how='left')

# drop rows without a global p95 threshold (cannot evaluate anomaly)
df_valid = df_merged[~df_merged['p95_threshold'].isna()].copy()

# ensure numeric
df_valid['value'] = pd.to_numeric(df_valid['value'], errors='coerce')
df_valid['p95_threshold'] = pd.to_numeric(df_valid['p95_threshold'], errors='coerce').fillna(0)

# mark anomaly if value >= p95_threshold
df_valid['is_anomaly'] = df_valid['value'] >= df_valid['p95_threshold']

# keep only anomalous rows
trace_anom_rows = df_valid[df_valid['is_anomaly']].copy()

# Aggregate anomalous trace rows by cmdb_id and trace_name
if not trace_anom_rows.empty:
    agg = (
        trace_anom_rows
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'size'),
            earliest_anomaly_timestamp=('dt', 'min'),
            max_value_within_window=('value', 'max'),
            p95_threshold=('p95_threshold', 'first')
        )
    )
    # compute deviation_ratio with special handling
    def compute_ratio(row):
        p95 = row['p95_threshold']
        vmax = row['max_value_within_window']
        if p95 == 0:
            return np.inf if vmax > 0 else 1.0
        else:
            return vmax / p95

    agg['deviation_ratio'] = agg.apply(compute_ratio, axis=1)

    # sort by earliest_anomaly_timestamp ascending to see propagation order and take top 20
    trace_anom_top20 = agg.sort_values('earliest_anomaly_timestamp', ascending=True).head(20).copy()

    # select and order requested columns
    trace_anom_top20 = trace_anom_top20[['cmdb_id', 'trace_name', 'anomaly_count',
                                         'earliest_anomaly_timestamp', 'max_value_within_window',
                                         'p95_threshold', 'deviation_ratio']]
else:
    trace_anom_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count',
                                             'earliest_anomaly_timestamp','max_value_within_window',
                                             'p95_threshold','deviation_ratio'])

# Output compact result (variable)
trace_anom_top20

```
Out[5]:
```
```python
# Summarize the trace anomaly aggregation (reuse trace_anom_top20)
try:
    df = trace_anom_top20.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No anomalous traces found for the selected components/time window."
else:
    total_groups = int(df.shape[0])
    overall_earliest = df['earliest_anomaly_timestamp'].min()
    lines = [
        f"Trace anomalies (window 2021-03-04 13:30–14:00 UTC+8): {total_groups} anomalous (cmdb_id, trace_name) groups found.",
        f"Earliest anomaly timestamp (UTC+8): {overall_earliest}.",
        "Top entries by propagation order (earliest first):"
    ]
    # show up to 8 for context
    for r in df.head(8).itertuples(index=False):
        cmdb = r.cmdb_id
        tname = r.trace_name
        cnt = int(r.anomaly_count)
        earliest = r.earliest_anomaly_timestamp
        vmax = float(r.max_value_within_window)
        p95 = float(r.p95_threshold)
        ratio = float(r.deviation_ratio)
        tshort = (tname[:80] + '...') if len(tname) > 80 else tname
        lines.append(f" - {earliest} | {cmdb} | {tshort} | count={cnt} | max={vmax:g} | p95={p95:g} | ratio={ratio:g}")
    lines.append("Observation: Most early anomalies originate from MG01 and MG02 (multiple calls to dockerA/B services) at ~13:30, followed by a couple of Tomcat traces. Deviation ratios are modest (≈1.03–1.91), indicating latency increases rather than extreme spikes.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  anomaly_count earliest_anomaly_timestamp  max_value_within_window  p95_threshold  deviation_ratio
70       MG01   trace.to_dockerA2.duration_p95              2  2021-03-04 13:30:00+08:00                 6.081200       3.625350         1.677410
117      MG02  trace.to_dockerB2.duration_mean              2  2021-03-04 13:30:00+08:00                 2.439865       1.283238         1.901334
118      MG02   trace.to_dockerB2.duration_p95              2  2021-03-04 13:30:00+08:00                 9.028650       8.291370         1.088921
112      MG02   trace.to_dockerA2.duration_p95              2  2021-03-04 13:30:00+08:00                 5.210500       3.439040         1.515103
111      MG02  trace.to_dockerA2.duration_mean              2  2021-03-04 13:30:00+08:00                 1.308353       0.713151         1.834608
109      MG02   trace.to_dockerA1.duration_p95              2  2021-03-04 13:30:00+08:00                 5.179550       3.738560         1.385440
108      MG02  trace.to_dockerA1.duration_mean              2  2021-03-04 13:30:00+08:00                 1.208703       0.897583         1.346620
75       MG01  trace.to_dockerB2.duration_mean              2  2021-03-04 13:30:00+08:00                 1.705493       1.585724         1.075529
114      MG02  trace.to_dockerB1.duration_mean              2  2021-03-04 13:30:00+08:00                 2.035549       1.674401         1.215688
126  Tomcat01    trace.from_root.duration_mean              1  2021-03-04 13:30:00+08:00                 0.796000       0.586346         1.357560
73       MG01   trace.to_dockerB1.duration_p95              2  2021-03-04 13:30:00+08:00                16.638050      10.335760         1.609756
72       MG01  trace.to_dockerB1.duration_mean              2  2021-03-04 13:30:00+08:00                 2.916556       2.559697         1.139415
69       MG01  trace.to_dockerA2.duration_mean              2  2021-03-04 13:30:00+08:00                 1.742875       0.912676         1.909632
67       MG01   trace.to_dockerA1.duration_p95              2  2021-03-04 13:30:00+08:00                 4.946600       3.655510         1.353190
66       MG01  trace.to_dockerA1.duration_mean              2  2021-03-04 13:30:00+08:00                 1.141184       0.877091         1.301102
127  Tomcat01     trace.from_root.duration_p95              1  2021-03-04 13:30:00+08:00                 2.149200       1.666680         1.289510
115      MG02   trace.to_dockerB1.duration_p95              2  2021-03-04 13:30:00+08:00                 8.480000       8.210580         1.032814
104      MG02        trace.from_root.row_count              1  2021-03-04 13:32:00+08:00               194.000000     190.400000         1.018908
71       MG01      trace.to_dockerA2.row_count              2  2021-03-04 13:32:00+08:00                94.000000      86.800000         1.082949
62       MG01        trace.from_root.row_count              1  2021-03-04 13:32:00+08:00               213.000000     201.500000         1.057072```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse existing variables if available
try:
    trace_agg
except NameError:
    raise RuntimeError("trace_agg (global trace aggregates) not found in environment.")

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Search trace_agg for trace_name containing 'mysql' (case-insensitive)
mysql_mask = trace_agg['trace_name'].str.contains('mysql', case=False, na=False)
mysql_trace_search = trace_agg[mysql_mask].copy()

# Select requested columns and sort by count desc, limit 50
select_cols = ['cmdb_id', 'trace_name', 'count', 'p50', 'p95', 'min', 'max']
# Some trace_agg variants might have slightly different column names; ensure existence
available_cols = [c for c in select_cols if c in mysql_trace_search.columns]
mysql_trace_search = mysql_trace_search[available_cols].sort_values('count', ascending=False).head(50)

# If matches found, filter df_trace to those (cmdb_id, trace_name) pairs within the incident window
if not mysql_trace_search.empty:
    # build set of pairs
    pairs = set(zip(mysql_trace_search['cmdb_id'], mysql_trace_search['trace_name']))
    # keep rows matching any pair
    mask_pairs = df_trace.apply(lambda r: (r['cmdb_id'], r['trace_name']) in pairs, axis=1)
    df_pairs = df_trace[mask_pairs].copy()
    # convert timestamp to timezone-aware datetime in UTC+8
    df_pairs['dt'] = pd.to_datetime(df_pairs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # incident window inclusive
    start = tz.localize(datetime(2021, 3, 4, 13, 30))
    end = tz.localize(datetime(2021, 3, 4, 14, 0))
    df_window = df_pairs[(df_pairs['dt'] >= start) & (df_pairs['dt'] <= end)].copy()
    if df_window.empty:
        # no rows in time window
        mysql_trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count',
                                                 'earliest_anomaly_timestamp','max_value_within_window',
                                                 'p95_threshold','deviation_ratio'])
    else:
        # attach global p95 threshold from trace_agg
        thresholds = trace_agg[['cmdb_id','trace_name','p95']].rename(columns={'p95':'p95_threshold'})
        df_merged = pd.merge(df_window, thresholds, on=['cmdb_id','trace_name'], how='left')
        # keep only rows with a p95_threshold (should exist)
        df_merged = df_merged[~df_merged['p95_threshold'].isna()].copy()
        # ensure numeric
        df_merged['value'] = pd.to_numeric(df_merged['value'], errors='coerce')
        df_merged['p95_threshold'] = pd.to_numeric(df_merged['p95_threshold'], errors='coerce').fillna(0)
        # mark anomaly if value >= p95
        df_merged['is_anomaly'] = df_merged['value'] >= df_merged['p95_threshold']
        trace_anom_rows = df_merged[df_merged['is_anomaly']].copy()
        if trace_anom_rows.empty:
            mysql_trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count',
                                                     'earliest_anomaly_timestamp','max_value_within_window',
                                                     'p95_threshold','deviation_ratio'])
        else:
            agg = (
                trace_anom_rows
                .groupby(['cmdb_id','trace_name'], as_index=False)
                .agg(
                    anomaly_count=('is_anomaly', 'size'),
                    earliest_anomaly_timestamp=('dt', 'min'),
                    max_value_within_window=('value', 'max'),
                    p95_threshold=('p95_threshold', 'first')
                )
            )
            # compute deviation_ratio with rules
            p95 = agg['p95_threshold'].to_numpy(dtype=float)
            vmax = agg['max_value_within_window'].to_numpy(dtype=float)
            ratio = np.where(p95 == 0, np.where(vmax > 0, np.inf, 1.0), vmax / p95)
            agg['deviation_ratio'] = ratio
            # sort by earliest_anomaly_timestamp ascending and limit 20
            mysql_trace_anoms = agg.sort_values('earliest_anomaly_timestamp', ascending=True).head(20).copy()
else:
    # no matches found: return empty search result and empty anomalies
    mysql_trace_search = mysql_trace_search  # empty
    mysql_trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count',
                                             'earliest_anomaly_timestamp','max_value_within_window',
                                             'p95_threshold','deviation_ratio'])

# Return compact outputs (variables)
mysql_trace_search, mysql_trace_anoms

```
Out[6]:
```
```python
# Summarize the search and anomaly results for 'mysql' traces in plain English.
import pandas as pd

# Use existing variables if present
search_empty = True
anoms_empty = True
if 'mysql_trace_search' in globals():
    try:
        search_empty = mysql_trace_search.empty
    except:
        search_empty = True
if 'mysql_trace_anoms' in globals():
    try:
        anoms_empty = mysql_trace_anoms.empty
    except:
        anoms_empty = True

lines = []
if search_empty:
    lines.append("Search: No trace_name containing 'mysql' found in trace_agg (global trace aggregates).")
    lines.append("Result: No MySQL-related trace groups to inspect; no anomalies detected for MySQL in the incident window.")
else:
    lines.append("Search: Found trace_name(s) containing 'mysql' in trace_agg.")
    if anoms_empty:
        lines.append("However, filtering trace.csv to the incident window produced no anomalous rows for those pairs.")
    else:
        lines.append("Anomalous trace groups (limited to top 20 by earliest timestamp) were detected for MySQL-related traces in the window.")

lines.append("Window: 2021-03-04 13:30–14:00 (UTC+8).")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, count, p50, p95, min, max]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, anomaly_count, earliest_anomaly_timestamp, max_value_within_window, p95_threshold, deviation_ratio]
Index: [])```
```